# Récupération d'un corpus d'artiste

L'objectif de ce notebook

In [1]:
from function.utils import os, tqdm, json

PATH_JSON = "../data/json_file/"

if not os.path.exists(PATH_JSON):
    os.makedirs(PATH_JSON)

In [2]:
from function.scrapping import get_url, scrapping_artiste
from datetime import datetime

start = 1950
current_year = datetime.now().year
years = range(start, current_year, 1)

PATH_ARTISTE = PATH_JSON+"dictionnaire_artiste.json"

headers = {    
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"}

URL_BASE = "https://hit-parade.net/"

liste_pages = list()
liste_artiste = list()

for year in tqdm.tqdm(years):
    liste_pages.extend(get_url(URL_BASE, str(year), headers))

for URL in tqdm.tqdm(liste_pages):
    liste_artiste.extend(scrapping_artiste(URL, headers))

    
list_new_artiste = set(liste_artiste)  


100%|██████████| 202/202 [01:00<00:00,  3.36it/s]


In [3]:
if os.path.exists(PATH_ARTISTE):
    with open(PATH_ARTISTE, "r") as f:
        ARTISTE = json.load(f)

nouveau_ARTISTE = [new_artiste for new_artiste in list_new_artiste if not new_artiste in ARTISTE]
ARTISTE = ARTISTE + nouveau_ARTISTE

In [4]:
import re
artiste_en_featuring = []

for artiste in ARTISTE:
    match = re.search(r"feat\.?|ft\.?|&", artiste)
    if match:
        featuring = re.split(r"feat | ft\. | &", artiste)
        artiste_en_featuring.extend([chanteur.strip() for chanteur in featuring])

ARTISTE = ARTISTE + artiste_en_featuring

ARTISTE = [
    x.strip()
    for x in ARTISTE
    if isinstance(x, str)
    and not re.search(r"feat\.?|ft\.?|&", x)
    and x.strip()
]

ARTISTE = list(set(ARTISTE))


In [5]:

with open(PATH_JSON+"dictionnaire_artiste2.json", "w", encoding="UTF-8") as json_file:
    json.dump(list(ARTISTE), json_file)